# Round 4 · Reports, shared intelligence and campaign takedown

**The attacker in the data:** a dispersed crew. One or two targets per account, ordinary hours, aged
accounts, half of them never post a link on the platform. No single account looks wrong.

What a crew cannot avoid is being a crew: operators share phones, payouts land in the same handles, and
victims eventually report. A partner bank has shared the handles and domains its customers complained
about. Intel is a lead, not a verdict.

**Your job:** add to your Round 2 flags in `out/suspects_r4.csv` (cumulative). Rank by score: the
checkpoint also judges your top 25, the queue an analyst works first.

In [ ]:
import sys
sys.path.insert(0, "../tools")
import pandas as pd
import trailmates as tm
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

import networkx as nx
F = tm.frames()
mem, dms, pays, reps = F["members"], F["dms"], F["payments"], F["reports"]
round2 = tm.read_suspects("../out/suspects_r2.csv")
intel = pd.DataFrame(tm.intel())
listed = set(intel["value"])
intel

In [ ]:
# seeds: evidence of scam. The starter uses one kind: members who mentioned a listed handle or domain.
sharers = set(dms.loc[dms["link"].isin(listed) | dms["payout_ref"].isin(listed), "sender"])
scam_reports = reps[reps["reason"] == "scam"].groupby("reported")["reporter"].nunique()
print(len(sharers), "members mentioned a listed indicator;",
      (scam_reports >= 2).sum(), "members were reported for a scam by two or more people")

In [ ]:
# association edges: a shared device, a shared non-community link domain
G = nx.Graph()
G.add_nodes_from(mem["member_id"])
for dev, ms in mem.groupby("device_id")["member_id"]:
    ms = list(ms)
    G.add_edges_from((ms[0], x) for x in ms[1:])
genuine = set(tm.world()["genuine_domains"])
for dom, ms in dms[dms["link"].notna() & ~dms["link"].isin(genuine)].groupby("link")["sender"]:
    ms = sorted(set(ms))
    G.add_edges_from((ms[0], x) for x in ms[1:])
print(G.number_of_edges(), "association edges")

In [ ]:
def round4(seeds, graph):
    # TODO (Round 4): the starter flags the seeds only. Add the other evidence (two or more scam reports;
    # payers of a listed handle whose most-talked-to new contact is the same member), then take down the
    # connected component around each seed. Never expand from a component without a seed, and be
    # suspicious of a component bigger than any crew could be.
    return {m: 2.0 for m in seeds}


new = {m: s for m, s in round4(sharers, G).items() if m not in round2}
scores = dict(round2)
scores.update(new)
tm.write_suspects("../out/suspects_r4.csv", scores)
print(len(new), "added;", len(scores), "in total")

In [ ]:
!{sys.executable} ../tools/checkpoint.py --round 4

## Read the scoreboard

- How big are the components around your seeds? `[len(c) for c in nx.connected_components(G) if c & set(sharers)]`
- Which listed indicator belongs to someone honest? What did the bank's note say about it?
- Follow the money: for each listed handle, count the `offplatform_contact` of its payers.

In [ ]:
listed_pays = pays[pays["payee"].isin(listed) & pays["offplatform_contact"].notna()]
listed_pays.groupby(["payee", "offplatform_contact"]).size().sort_values(ascending=False).head(15)

PASS? Go to `05_reveal.ipynb`. Stuck? `python tools/checkpoint.py --round 4 --catch-up`.